# Μάθημα 05 - Agentic RAG


## Setup

Αυτό το τετράδιο παρουσιάζει το μοτίβο Agentic RAG (Retrieval-Augmented Generation) χρησιμοποιώντας το Microsoft Agent Framework.

**Προαπαιτούμενα:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — το endpoint της υπηρεσίας Azure AI Search σας
- `AZURE_SEARCH_API_KEY` — το κλειδί API της υπηρεσίας Azure AI Search σας
- Ανάπτυξη Azure OpenAI ρυθμισμένη μέσω μεταβλητών περιβάλλοντος
- Αυθεντικοποίηση Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Τι είναι το Agentic RAG;

Το παραδοσιακό RAG ακολουθεί μια σταθερή ροή εργασιών: ανακτά έγγραφα και στη συνέχεια δημιουργεί μια απάντηση. Το **Agentic RAG** πηγαίνει ένα βήμα παραπέρα δίνοντας στον πράκτορα την αυτονομία να αποφασίζει **πότε** και **πώς** θα ανακτήσει πληροφορίες.

Με το Agentic RAG, ο πράκτορας μπορεί:
- να **αποφασίσει** εάν είναι απαραίτητη η ανάκτηση πριν απαντήσει σε μια ερώτηση
- να **επιλέξει** ποια πηγή δεδομένων ή εργαλείο θα χρησιμοποιήσει για τη λήψη πληροφοριών
- να **αξιολογήσει** τα αποτελέσματα της ανάκτησης και να πραγματοποιήσει περαιτέρω αναζητήσεις αν η πρώτη προσπάθεια είναι ανεπαρκής
- να **συνδυάσει** πληροφορίες από πολλαπλά βήματα ανάκτησης σε μια συνεκτική απάντηση

Αυτό καθιστά τον πράκτορα πιο ευέλικτο και ακριβή σε σύγκριση με μια στατική ακολουθία ανάκτησης και δημιουργίας απάντησης.


## Δημιουργία Εργαλείου Αναζήτησης

Στο Agentic RAG, οι εξωτερικές πηγές δεδομένων περιβάλλονται ως **εργαλεία** που ο πράκτορας μπορεί να ενεργοποιήσει κατόπιν ζήτησης. Αυτό επιτρέπει στον πράκτορα να θεωρεί την ανάκτηση απλώς ως μία ακόμη ενέργεια που μπορεί να εκτελέσει, αντί για ένα υποχρεωτικό βήμα.

Παρακάτω ορίζουμε μια βάση γνώσεων ταξιδιών και την εκθέτουμε ως εργαλείο που ο πράκτορας μπορεί να καλέσει για να αναζητήσει πληροφορίες προορισμών.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Κατασκευή του Πράκτορα RAG

Τώρα δημιουργούμε έναν πράκτορα που έχει εντολή να **ανακτά πάντα πληροφορίες πριν απαντήσει**. Ο πράκτορας χρησιμοποιεί το εργαλείο `search_travel_knowledge` για να βασίσει τις απαντήσεις του στη βάση γνώσεων αντί να στηρίζεται στα δικά του δεδομένα εκπαίδευσης.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Επαναληπτική Ανάκτηση — Το Πρότυπο Κατασκευαστή-Ελεγκτή

Ένα βασικό πλεονέκτημα του Agentic RAG είναι η **επαναληπτική ανάκτηση**. Ο πράκτορας μπορεί να εκτελέσει πολλούς γύρους αναζήτησης για να επαληθεύσει, να βελτιώσει ή να επεκτείνει τα αρχικά του ευρήματα — παρόμοια με μια ροή εργασίας "κατασκευαστή-ελεγκτή":

1. **Βήμα κατασκευαστή**: Ο πράκτορας ανάγει αρχικές πληροφορίες και συντάσσει μια απάντηση.
2. **Βήμα ελεγκτή**: Ο πράκτορας εκτελεί επιπλέον ανακτήσεις για να επαληθεύσει λεπτομέρειες ή να καλύψει κενά.

Παρακάτω, ο πράκτορας ζητείται να απαντήσει σε μια ερώτηση που απαιτεί σύγκριση πολλών προορισμών, προκαλώντας τον να αναζητήσει αρκετές φορές.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να δημιουργήσετε ένα σύστημα **Agentic RAG** χρησιμοποιώντας το Microsoft Agent Framework:

- Το **Agentic RAG** επιτρέπει στους πράκτορες να αποφασίζουν αυτόνομα πότε θα ανακτήσουν πληροφορίες, καθιστώντας την ανάκτηση δυναμική και όχι σταθερή.
- **Εργαλεία ως πηγές δεδομένων**: Εξωτερικές βάσεις γνώσης (όπως το Azure AI Search) περιτυλίγονται ως εργαλεία που μπορεί να επικαλεστεί ο πράκτορας.
- **Επαναληπτική ανάκτηση**: Το μοτίβο maker-checker επιτρέπει στον πράκτορα να πραγματοποιεί πολλούς γύρους ανάκτησης — αναζήτηση, επαλήθευση και βελτίωση — πριν παράξει μια τελική απάντηση.

Σε παραγωγή, θα αντικαθιστούσατε τη μνήμη `TRAVEL_KNOWLEDGE_BASE` με έναν πραγματικό δείκτη Azure AI Search για να διαχειριστείτε την ανάκτηση μεγάλου όγκου ταξιδιωτικών εγγράφων.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με Τεχνητή Νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα του θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
